# Day 6: Forward & Backward Propagation From Scratch

**Module 2 — Neural Network Basics | 100 Days of Data Science**

## Why This Matters
This is the day everything from Module 1 and the last two days finally clicks together: dot products (Day 1), the chain rule (Day 2), cross-entropy (Day 3), the MLP architecture (Day 4), and activation functions (Day 5) all combine into **training** — the actual process by which a neural network learns.

Today we train an MLP to solve XOR **from scratch**, with no framework, so you see every gradient calculation explicitly. Then we do the same thing in 10 lines of PyTorch.

## Topics Covered Today
1. Forward propagation (recap, now formalized layer by layer)
2. Loss computation
3. Backward propagation — deriving every gradient by hand
4. The full training loop, from scratch, solving XOR
5. The same training loop in PyTorch
6. Practice exercises

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
print("NumPy version:", np.__version__)

---
## 1. The Network We're Training

Architecture: **2 inputs → 2 hidden neurons (sigmoid) → 1 output neuron (sigmoid)**

Task: learn the XOR function (the exact problem a single perceptron couldn't solve, from Day 4).

**Forward pass equations:**
$$z_1 = X W_1 + b_1, \quad a_1 = \sigma(z_1)$$
$$z_2 = a_1 W_2 + b_2, \quad a_2 = \sigma(z_2) = \hat{y}$$

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(a):
    # NOTE: takes the *activated* value a = sigmoid(z), since sigmoid'(z) = a*(1-a)
    return a * (1 - a)

# XOR dataset
X = np.array([[0,0],[0,1],[1,0],[1,1]])
y = np.array([[0],[1],[1],[0]])

print("X:\n", X)
print("y:\n", y)

---
## 2. Loss Function

We'll use **Mean Squared Error (MSE)** here for simplicity of derivation (cross-entropy from Day 3 works too, and is more standard, but MSE gradients are easier to trace by hand for a first pass):

$$L = \frac{1}{n}\sum (\hat{y} - y)^2$$

In [ ]:
def mse_loss(y_true, y_pred):
    return np.mean((y_pred - y_true) ** 2)

---
## 3. Backward Propagation — Deriving Every Gradient

We need $\frac{\partial L}{\partial W_1}, \frac{\partial L}{\partial b_1}, \frac{\partial L}{\partial W_2}, \frac{\partial L}{\partial b_2}$. Using the chain rule (Day 2), we work **backward** from the loss:

**Output layer:**
$$\delta_2 = \frac{\partial L}{\partial a_2} \cdot \frac{\partial a_2}{\partial z_2} = 2(\hat{y}-y) \cdot \sigma'(z_2)$$
$$\frac{\partial L}{\partial W_2} = a_1^T \delta_2, \qquad \frac{\partial L}{\partial b_2} = \sum \delta_2$$

**Hidden layer (error flows backward through $W_2$):**
$$\delta_1 = (\delta_2 W_2^T) \cdot \sigma'(z_1)$$
$$\frac{\partial L}{\partial W_1} = X^T \delta_1, \qquad \frac{\partial L}{\partial b_1} = \sum \delta_1$$

This backward flow of `delta` terms IS backpropagation — each layer passes its error signal to the layer before it, exactly like the chain rule example from Day 2.

In [ ]:
class MLPFromScratch:
    def __init__(self, input_size, hidden_size, output_size):
        self.W1 = np.random.randn(input_size, hidden_size) * 0.5
        self.b1 = np.zeros((1, hidden_size))
        self.W2 = np.random.randn(hidden_size, output_size) * 0.5
        self.b2 = np.zeros((1, output_size))

    def forward(self, X):
        self.z1 = X @ self.W1 + self.b1
        self.a1 = sigmoid(self.z1)
        self.z2 = self.a1 @ self.W2 + self.b2
        self.a2 = sigmoid(self.z2)
        return self.a2

    def backward(self, X, y, lr=0.5):
        n = X.shape[0]
        y_pred = self.a2

        # Output layer gradients
        d_loss_a2 = 2 * (y_pred - y) / n
        delta2 = d_loss_a2 * sigmoid_derivative(self.a2)
        dW2 = self.a1.T @ delta2
        db2 = np.sum(delta2, axis=0, keepdims=True)

        # Hidden layer gradients (error propagated backward through W2)
        delta1 = (delta2 @ self.W2.T) * sigmoid_derivative(self.a1)
        dW1 = X.T @ delta1
        db1 = np.sum(delta1, axis=0, keepdims=True)

        # Gradient descent update (Day 2)
        self.W2 -= lr * dW2
        self.b2 -= lr * db2
        self.W1 -= lr * dW1
        self.b1 -= lr * db1

---
## 4. Full Training Loop — Solving XOR From Scratch

In [ ]:
model = MLPFromScratch(input_size=2, hidden_size=4, output_size=1)

epochs = 5000
losses = []

for epoch in range(epochs):
    y_pred = model.forward(X)
    loss = mse_loss(y, y_pred)
    losses.append(loss)
    model.backward(X, y, lr=0.5)

    if epoch % 1000 == 0:
        print(f"Epoch {epoch:4d} | Loss: {loss:.6f}")

print("\nFinal predictions:")
final_pred = model.forward(X)
for inputs, true_val, pred in zip(X, y, final_pred):
    print(f"{inputs} -> predicted: {pred[0]:.4f} (rounded: {round(pred[0])}), true: {true_val[0]}")

In [ ]:
plt.figure(figsize=(6,4))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Training Loss Over Time -- The Network Learning XOR')
plt.grid(True)
plt.show()

**What just happened:** starting from random weights, backpropagation computed exactly how much each weight contributed to the error, and gradient descent nudged every weight to reduce it — repeated 5000 times until the network learned to solve a problem that's impossible for a single perceptron.

---
## 5. The Same Training Loop in PyTorch

This is why frameworks exist — autograd (Day 2) computes all these gradients automatically. Same problem, same architecture, ~15 lines.

In [ ]:
try:
    import torch
    import torch.nn as nn

    torch.manual_seed(42)

    X_t = torch.tensor(X, dtype=torch.float32)
    y_t = torch.tensor(y, dtype=torch.float32)

    model_torch = nn.Sequential(
        nn.Linear(2, 4),
        nn.Sigmoid(),
        nn.Linear(4, 1),
        nn.Sigmoid()
    )

    criterion = nn.MSELoss()
    optimizer = torch.optim.SGD(model_torch.parameters(), lr=0.5)

    for epoch in range(5000):
        y_pred = model_torch(X_t)
        loss = criterion(y_pred, y_t)

        optimizer.zero_grad()
        loss.backward()   # autograd computes ALL gradients automatically
        optimizer.step()  # applies the gradient descent update

        if epoch % 1000 == 0:
            print(f"Epoch {epoch:4d} | Loss: {loss.item():.6f}")

    print("\nFinal predictions:")
    with torch.no_grad():
        final = model_torch(X_t)
        for inputs, true_val, pred in zip(X, y, final):
            print(f"{inputs} -> predicted: {pred.item():.4f} (rounded: {round(pred.item())}), true: {true_val[0]}")
except ImportError:
    print("PyTorch not installed. Run: pip install torch")

---
## 6. Practice Exercises
Try these before Day 7:

1. Change `hidden_size` in `MLPFromScratch` from 4 to 2. Does it still converge? How many epochs does it take?
2. Change the learning rate to `0.05` and `5.0`. What happens to the loss curve in each case?
3. Replace MSE loss with cross-entropy loss in the from-scratch model — you'll need to derive the new `delta2` formula (hint: for sigmoid + cross-entropy, it simplifies to just `y_pred - y_true`).
4. In the PyTorch version, swap `nn.Sigmoid()` in the hidden layer for `nn.ReLU()` and compare convergence speed.
5. In your own words: explain what `delta1 = (delta2 @ W2.T) * sigmoid_derivative(a1)` is doing — why do we multiply by `W2.T`?

In [ ]:
# Your practice code here


---
## Summary
- **Forward propagation**: input flows through layers, producing a prediction
- **Loss**: measures how wrong that prediction is
- **Backward propagation**: the chain rule applied layer by layer, computing how much each weight contributed to the error
- **Gradient descent**: uses those gradients to update every weight, reducing loss over many epochs
- **Autograd** (PyTorch) automates the entire backward pass — but now you know exactly what it's doing under the hood

Next up: **Day 7 — Loss Functions & Optimizers (SGD, Adam, RMSprop)**

---
*Part of the 100 Days of Data Science series | DL-for-Data-Science repo*